# Análisis exploratorio y segmentación — Mall Customers

**Gestión de Proyectos de Datos (SCY1102) · Evaluación Parcial 3 (EP5)**

Este notebook documenta la exploración del dataset, el tratamiento de las categorías `Unknown`, la selección de `k`, el modelo K-Means y la revisión de la distribución de los clusters por género. El notebook y el dashboard utilizan la misma función definida en `src/model.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_customers
from src.model import (
    MODEL_FEATURES,
    N_INIT,
    RANDOM_STATE,
    choose_k,
    cluster_profile,
    evaluate_k_candidates,
    fit_customer_clusters,
    gender_cluster_audit,
)

customers = load_customers()
print(f"Dataset cargado: {customers.shape[0]} clientes · {customers.shape[1]} variables preparadas")
customers.head()

Dataset cargado: 200 clientes · 9 variables preparadas


,customer_id,gender,age,education,marital_status,annual_income_k,spending_score,gender_label,has_unknown
0,1,M,19,High School,Married,15,39,Hombre,False
1,2,M,21,Graduate,Single,15,81,Hombre,False
2,3,F,20,Graduate,Married,16,6,Mujer,False
3,4,F,23,High School,Unknown,16,77,Mujer,True
4,5,F,31,Uneducated,Married,17,40,Mujer,False


## 1. Exploración inicial

Se revisan tamaño, variables y estadísticos descriptivos antes del modelado. El dataset original contiene 200 clientes y siete variables: identificador, género, edad, educación, estado civil, ingreso anual y Spending Score.

In [2]:
print("Dimensiones:", customers.shape)
print("\nTipos de datos:")
display(customers[["customer_id", "gender", "age", "education", "marital_status", "annual_income_k", "spending_score"]].dtypes.to_frame("dtype"))

print("\nResumen de variables numéricas:")
display(customers[["age", "annual_income_k", "spending_score"]].describe().round(2))

Dimensiones: (200, 9)

Tipos de datos:


,dtype
customer_id,int64
gender,object
age,int64
education,object
marital_status,object
annual_income_k,int64
spending_score,int64



Resumen de variables numéricas:


,age,annual_income_k,spending_score
count,200.00,200.00,200.00
mean,38.85,60.56,50.20
std,13.97,26.26,25.82
min,18.00,15.00,1.00
25%,28.75,41.50,34.75
50%,36.00,61.50,50.00
75%,49.00,78.00,73.00
max,70.00,137.00,99.00


In [3]:
fig = px.scatter(
    customers,
    x="annual_income_k",
    y="spending_score",
    color="gender_label",
    hover_data=["age", "education", "marital_status"],
    title="Ingreso anual vs. Spending Score",
    labels={
        "annual_income_k": "Ingreso anual (k$)",
        "spending_score": "Spending Score",
        "gender_label": "Género",
    },
)
fig.show()

## 2. Tratamiento de `Unknown`

Las categorías `Unknown` se conservan y se documentan. **No se imputan ni se eliminan para entrenar K-Means**, porque las variables utilizadas por el modelo son únicamente edad, ingreso anual y Spending Score. De esta forma se evita descartar registros sin una justificación técnica y se mantiene la trazabilidad de los 200 clientes.

In [4]:
education_unknown = int(customers["education"].eq("Unknown").sum())
marital_unknown = int(customers["marital_status"].eq("Unknown").sum())
any_unknown = int(customers["has_unknown"].sum())

print(f"Unknown en Education:      {education_unknown}")
print(f"Unknown en Marital Status: {marital_unknown}")
print(f"Clientes con al menos un Unknown: {any_unknown}")

print("\nDistribución de Education:")
display(customers["education"].value_counts().to_frame("Clientes"))
print("\nDistribución de Marital Status:")
display(customers["marital_status"].value_counts().to_frame("Clientes"))

Unknown en Education:      26
Unknown en Marital Status: 15
Clientes con al menos un Unknown: 39

Distribución de Education:


,Clientes
education,
Graduate,67
High School,40
Uneducated,31
Unknown,26
College,17
Doctorate,11
Post-Graduate,8



Distribución de Marital Status:


,Clientes
marital_status,
Married,112
Single,61
Unknown,15
Divorced,12


## 3. Variables y parámetros del modelo

K-Means se aplica sobre **edad, ingreso anual y Spending Score estandarizados**. La estandarización evita que una variable domine la distancia euclidiana solo por tener una escala numérica mayor.

Parámetros reproducibles del modelo:
- `random_state = 42`
- `n_init = 20`
- candidatos de `k`: 2 a 10
- variables: `age`, `annual_income_k`, `spending_score`

In [5]:
print("Variables del modelo:", MODEL_FEATURES)
print("random_state:", RANDOM_STATE)
print("n_init:", N_INIT)

Variables del modelo: ['age', 'annual_income_k', 'spending_score']
random_state: 42
n_init: 20


## 4. Selección de `k`: método del codo y coeficiente de silueta

Se evalúan valores de `k` entre 2 y 10. La **inercia** permite observar el método del codo y el **coeficiente de silueta** mide simultáneamente cohesión interna y separación entre grupos. En este caso, la curva del codo sugiere revisar la zona intermedia y la silueta permite resolver la selección final.

In [6]:
k_metrics = evaluate_k_candidates(customers)
display(k_metrics.round({"inertia": 3, "silhouette": 4}))

fig_inertia = px.line(
    k_metrics, x="k", y="inertia", markers=True,
    title="Método del codo — Inercia por número de clusters",
    labels={"k": "Número de clusters (k)", "inertia": "Inercia"},
)
fig_inertia.show()

fig_silhouette = px.line(
    k_metrics, x="k", y="silhouette", markers=True,
    title="Coeficiente de silueta por número de clusters",
    labels={"k": "Número de clusters (k)", "silhouette": "Coeficiente de silueta"},
)
fig_silhouette.show()

,k,inertia,silhouette
0,2,389.386,0.3355
1,3,295.212,0.3578
2,4,205.225,0.4040
3,5,168.248,0.4166
4,6,133.868,0.4274
5,7,117.012,0.4172
6,8,103.829,0.4087
7,9,92.349,0.4201
8,10,81.987,0.3973


In [7]:
selected_k, _ = choose_k(customers)
best = k_metrics.loc[k_metrics["k"].eq(selected_k)].iloc[0]
print(f"k seleccionado: {selected_k}")
print(f"Coeficiente de silueta: {best['silhouette']:.4f}")
print(f"Inercia: {best['inertia']:.3f}")

k seleccionado: 6
Coeficiente de silueta: 0.4274
Inercia: 133.868


**Decisión:** se utiliza `k = 6`. Entre los candidatos evaluados es el valor con mayor coeficiente de silueta; además, la reducción de inercia comienza a mostrar ganancias menores después de la zona de 4–6 clusters. Por ello, `k = 6` ofrece el mejor equilibrio observado entre separación y nivel de detalle para este dataset.

## 5. Modelo K-Means final

Se ajusta el modelo usando la función compartida `fit_customer_clusters()`. Esta es la **misma función que utiliza el dashboard**, asegurando que las asignaciones y parámetros sean consistentes entre ambos entregables.

In [8]:
clustered, model_info = fit_customer_clusters(customers, k=selected_k)
print("Parámetros del modelo final:")
for key, value in model_info.items():
    print(f"  {key}: {value}")

clustered[["customer_id", "age", "annual_income_k", "spending_score", "cluster"]].head(10)

Parámetros del modelo final:
  k: 6
  silhouette: 0.42742814991580175
  inertia: 133.86833362685584
  random_state: 42
  n_init: 20
  features: ['age', 'annual_income_k', 'spending_score']


,customer_id,age,annual_income_k,spending_score,cluster
0,1,19,15,39,Cluster 1
1,2,21,15,81,Cluster 1
2,3,20,16,6,Cluster 2
3,4,23,16,77,Cluster 1
4,5,31,17,40,Cluster 2
5,6,22,17,76,Cluster 1
6,7,35,18,6,Cluster 2
7,8,23,18,94,Cluster 1
8,9,64,19,3,Cluster 2
9,10,30,19,72,Cluster 1


## 6. Descripción de los clusters

Cada cluster se caracteriza mediante su **tamaño, edad promedio, ingreso anual promedio y Spending Score promedio**.

In [9]:
profile = cluster_profile(clustered)
display(profile.round({
    "Edad_promedio": 2,
    "Ingreso_promedio": 2,
    "Spending_promedio": 2,
}))

,cluster_id,cluster,Clientes,Edad_promedio,Ingreso_promedio,Spending_promedio
0,1,Cluster 1,24,25.25,25.83,76.92
1,2,Cluster 2,21,45.52,26.29,19.38
2,3,Cluster 3,45,56.33,54.27,49.07
3,4,Cluster 4,38,26.68,57.58,47.79
4,5,Cluster 5,39,32.69,86.54,82.13
5,6,Cluster 6,33,41.94,88.94,16.97


In [10]:
fig = px.scatter(
    clustered,
    x="annual_income_k",
    y="spending_score",
    color="cluster",
    hover_data=["age", "gender_label"],
    title="Clusters K-Means — Ingreso anual y Spending Score",
    labels={
        "annual_income_k": "Ingreso anual (k$)",
        "spending_score": "Spending Score",
        "cluster": "Cluster",
    },
)
fig.show()

## 7. Revisión de sesgos: distribución por género

El género **no se incluye como variable de entrenamiento**. La revisión siguiente no demuestra causalidad ni discriminación; sirve para verificar cómo se distribuyen mujeres y hombres después de formar los clusters con edad, ingreso y Spending Score.

In [11]:
gender_audit = gender_cluster_audit(clustered)
display(gender_audit.round({"Porcentaje": 1}))

fig = px.bar(
    gender_audit,
    x="cluster",
    y="Porcentaje",
    color="gender_label",
    barmode="stack",
    text=gender_audit["Porcentaje"].map(lambda value: f"{value:.1f}%"),
    title="Composición de género dentro de cada cluster",
    labels={"cluster": "Cluster", "Porcentaje": "Porcentaje", "gender_label": "Género"},
)
fig.update_yaxes(range=[0, 100])
fig.show()

,cluster_id,cluster,gender_label,Clientes,Porcentaje
0,1,Cluster 1,Hombre,10,41.7
1,1,Cluster 1,Mujer,14,58.3
2,2,Cluster 2,Hombre,8,38.1
3,2,Cluster 2,Mujer,13,61.9
4,3,Cluster 3,Hombre,19,42.2
5,3,Cluster 3,Mujer,26,57.8
6,4,Cluster 4,Hombre,14,36.8
7,4,Cluster 4,Mujer,24,63.2
8,5,Cluster 5,Hombre,18,46.2
9,5,Cluster 5,Mujer,21,53.8


In [12]:
overall_gender = (
    clustered["gender_label"].value_counts(normalize=True).mul(100).round(1).rename("Porcentaje")
)
print("Distribución global de género:")
display(overall_gender.to_frame())

Distribución global de género:


,Porcentaje
gender_label,
Mujer,56.0
Hombre,44.0


### Limitaciones de la revisión de sesgos

- La muestra contiene solo **200 clientes**, por lo que la representatividad es limitada.
- El género no participa en el entrenamiento; observar diferencias entre clusters **no prueba por sí solo un sesgo discriminatorio**.
- Existen categorías `Unknown` en educación y estado civil. Aunque no intervienen en K-Means, limitan la caracterización completa de algunos clientes.
- K-Means presupone grupos relativamente compactos en términos de distancia euclidiana y puede ser sensible a la forma de los datos.
- Las tres variables fueron estandarizadas, por lo que reciben un peso comparable dentro de la distancia. Esa elección metodológica debe mantenerse documentada.
- Los clusters son perfiles analíticos del dataset disponible; no deben utilizarse automáticamente para excluir clientes o tomar decisiones comerciales sin supervisión humana y validación adicional.

## 8. Trazabilidad con el dashboard

El notebook y el dashboard importan el modelo desde `src/model.py`. Esto evita mantener dos implementaciones distintas y asegura que la selección de variables, estandarización, `random_state`, `n_init`, valor de `k` y asignación de clusters sean consistentes.